# BLUE SCORE

# -> It measure N-gram Precision instead of Relevance


## -> langugage transalation but not for Summariation


In [4]:
from collections import Counter
import math

def compute_blue(reference,generate_text):
    reference_tokens = reference.lower().split() #-> [i,am,a,boy]
    generate_tokens = generate_text.lower().split()

    reference_counts = Counter(reference_tokens) #-> {i,1:,am:1,a:1,boy:1}
    generate_counts = Counter(generate_tokens)

    overlap = sum((reference_counts & generate_counts).values())
    total_reference = sum(reference_counts.values())
    total_generate = sum(generate_counts.values())

    # count match tokens
    matches = 0
    for token, count in generate_counts.items():
        matches += min(count, reference_counts[token])

    precision = matches / total_generate if generate_counts else 0.0


    if len(generate_counts) > len(reference_counts):
        bp = 1.0

    else:
        bp = math.exp(1 - len(reference_counts) / len(generate_counts))

    

    return bp*precision

In [5]:
# ─── Test data: News article summarisation task ───────────────────────────────
reference = "The Federal Reserve raised interest rates by 0.25 percent to combat rising inflation"

candidates = {
"Agent A (close paraphrase)": "The Fed increased rates by 0.25 percent to fight inflation",
"Agent B (exact match)": "The Federal Reserve raised interest rates by 0.25 percent to combat rising inflation",
"Agent C (hallucination)": "The Federal Reserve cut rates by 1 percent to stimulate economic growth",
}


for name, cand in candidates.items():
    score = compute_blue(reference, cand)
    print(f"{name}: BLEU Score = {score:.4f}")

Agent A (close paraphrase): BLEU Score = 0.5186
Agent B (exact match): BLEU Score = 1.0000
Agent C (hallucination): BLEU Score = 0.5367


In [ ]:
## ROUGUE SCORE - Recall orientiend Understudy for Gisting Evaluation

## Developed for Summarization (2004)

ROUGE-N: Measures n-gram overlap between generated and reference text (e.g., ROUGE-1 for unigrams, ROUGE-2 for bigrams).
ROUGE-L: Measures longest common subsequence (LCS) between generated and reference text, capturing fluency and coherence.



In [7]:
!pip3 install rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=b21870ee296aa84cab82844389c48cc42db5a6d0177684a28af01e1b6afe49d6
  Stored in directory: /Users/cardinality/Library/Caches/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [11]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)


# Use a longer summarisation example
article = """
OpenAI released GPT-4 in March 2023, claiming it was significantly more capable
than GPT-3.5. The model demonstrated improved reasoning, reduced hallucinations,
and better instruction following. It passed the bar exam in the top 10 percentile.
"""

reference_summary = "OpenAI launched GPT-4 in March 2023, which showed improved reasoning and passed the bar exam in the top 10 percentile."

agent_summaries = {
    "Good summary": "OpenAI released GPT-4 in March 2023 with enhanced reasoning capabilities and passed the bar exam in the top 10 percentile.",
    "Missing key facts": "OpenAI released a new AI model that was more capable than previous versions.",
    "Verbatim copy": "OpenAI released GPT-4 in March 2023, claiming it was significantly more capable than GPT-3.5. The model demonstrated improved reasoning.",
    "Hallucinated summary": "OpenAI released GPT-4 in June 2023. The model failed the bar exam but excelled at creative writing tasks.",
}


In [12]:
for name, summary in agent_summaries.items():
    scores = scorer.score(reference_summary, summary)
    print(f"{name}: ROUGE-1={scores['rouge1'].fmeasure:.4f}, ROUGE-2={scores['rouge2'].fmeasure:.4f}, ROUGE-L={scores['rougeL'].fmeasure:.4f}")

Good summary: ROUGE-1=0.8095, ROUGE-2=0.6500, ROUGE-L=0.8095
Missing key facts: ROUGE-1=0.0588, ROUGE-2=0.0000, ROUGE-L=0.0588
Verbatim copy: ROUGE-1=0.4186, ROUGE-2=0.2439, ROUGE-L=0.3721
Hallucinated summary: ROUGE-1=0.4500, ROUGE-2=0.2105, ROUGE-L=0.4000


In [13]:
## BERT SCORE
!pip3 install bert-score



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from bert_score import score as bert_score

paraphrase_ref = ["The Federal Reserve raised interest rates by 0.25 percent to combat rising inflation"]

paraphrase_candidates = [
    "The Federal Reserve raised interest rates by 0.25 percent to combat rising inflation",  # Exact
    "The Fed increased rates by a quarter point to fight inflation",                          # Paraphrase
    "The central bank hiked borrowing costs to address price pressures",                      # Strong paraphrase  
    "The Federal Reserve cut rates by 0.25 percent to stimulate growth",                      # Factually wrong
]

In [ ]:
P, R, F1 = bert_score(paraphrase_candidates,paraphrase_ref*4,lang='en',verbose=True,model_type='bert-base-uncased')

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 10.77 seconds, 0.37 sentences/sec


In [19]:
for f1,cand in zip(F1,paraphrase_candidates):
    print(f"Candidate: '{cand}' - BERT Score F1: {f1:.4f}")

Candidate: 'The Federal Reserve raised interest rates by 0.25 percent to combat rising inflation' - BERT Score F1: 1.0000
Candidate: 'The Fed increased rates by a quarter point to fight inflation' - BERT Score F1: 0.9495
Candidate: 'The central bank hiked borrowing costs to address price pressures' - BERT Score F1: 0.8913
Candidate: 'The Federal Reserve cut rates by 0.25 percent to stimulate growth' - BERT Score F1: 0.9642


In [ ]:
## Hallucination/ Grounding

In [20]:
from sentence_transformers import SentenceTransformer, util
import torch


embedding_model = SentenceTransformer('all-MiniLM-L6-v2')


# Simulate RAG: these are the retrieved context chunks
retrieved_chunks = [
    "Tesla reported Q3 2024 revenue of $25.2 billion, representing an 8% year-over-year increase.",
    "The company delivered 435,059 vehicles in Q3 2024, slightly missing analyst expectations of 463,310.",
    "Tesla's gross margin for the automotive segment was 17.1%, down from 18.7% a year earlier.",
    "The Gigafactory Berlin achieved record production levels in Q3 2024.",
]

# Agent claims to evaluate
agent_claims = [
    "Tesla's Q3 2024 revenue was $25.2 billion, an 8% increase.",       # grounded
    "Tesla delivered 450,000 vehicles, exceeding analyst expectations.",  # hallucination
    "Tesla invested heavily in AI for autonomous driving.",               # not in context
    "Berlin factory had record production levels.",                       # grounded
]

README.md: 0.00B [00:00, ?B/s]

In [21]:
GROUNDED_THRESHOLD = 0.7

embeddings_chunks = embedding_model.encode(retrieved_chunks, convert_to_tensor=True)
for claim in   agent_claims:
    embeddings_claims = embedding_model.encode(claim, convert_to_tensor=True)
    similarities = util.cos_sim(embeddings_claims, embeddings_chunks)[0]

    best_similarity = torch.max(similarities).item()
    best_chunk_idx = torch.argmax(similarities).item()

    is_grounded = best_similarity >= GROUNDED_THRESHOLD
    grounding_status = "Grounded" if is_grounded else "Not Grounded"

    print(f"Claim: '{claim}' - {grounding_status} (Best similarity: {best_similarity:.4f} with chunk: '{retrieved_chunks[best_chunk_idx]}')")




Claim: 'Tesla's Q3 2024 revenue was $25.2 billion, an 8% increase.' - Grounded (Best similarity: 0.9422 with chunk: 'Tesla reported Q3 2024 revenue of $25.2 billion, representing an 8% year-over-year increase.')
Claim: 'Tesla delivered 450,000 vehicles, exceeding analyst expectations.' - Not Grounded (Best similarity: 0.6599 with chunk: 'The company delivered 435,059 vehicles in Q3 2024, slightly missing analyst expectations of 463,310.')
Claim: 'Tesla invested heavily in AI for autonomous driving.' - Not Grounded (Best similarity: 0.4670 with chunk: 'Tesla reported Q3 2024 revenue of $25.2 billion, representing an 8% year-over-year increase.')
Claim: 'Berlin factory had record production levels.' - Not Grounded (Best similarity: 0.6986 with chunk: 'The Gigafactory Berlin achieved record production levels in Q3 2024.')


BLUE - Transalation and Structure tempalate evaluation - Don't use in  QA task or summarization


ROUGUE - Summarization  - Don't use in Creative geneartion, Factual accuracy



BERT SCORE - Pharaphase task, Semantic Simialrity, - Don't use it when you need fast evaluation


Hallucinatio /Groundness - RAG Pipelines - DOn't use in Non-Retrival Agents


